# AIPerf Use Case 3 — Trace-Based Benchmarking with the Mooncake Trace

Replay **real production traffic** at an endpoint instead of synthetic prompts, using the open-sourced Mooncake arXiv-Q&A trace and `--custom-dataset-type mooncake_trace`.

These use-case notebooks demonstrate AIPerf capabilities directly.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — the Mooncake trace](#3-input--the-mooncake-trace)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)

## 1. Overview

Synthetic benchmarks (fixed or sampled ISL/OSL) are repeatable but unrealistic: every prompt is independent random tokens, so prefix/KV-cache reuse never happens. The Mooncake trace fixes that with a clever anonymization:

- Each record is a **timestamp + input/output lengths + block hash IDs** (each hash id stands for a fixed-size token block).
- **Repeated hash IDs across requests = shared prefixes** (e.g. follow-up questions about the same uploaded paper) — real reuse structure, zero user data.
- AIPerf expands the hashes into natural-language text **while preserving the repetition**, so a KV-cache-enabled server sees realistic cache hits.

Primary application: **A/B testing KV-cache optimizations** (e.g. Dynamo's KV-aware routing) — replay the same trace at two endpoints and compare TTFT. Synthetic prompts cannot show this difference at all.

> **Naming collision with this repo:** the Model Selection / Sizing scripts also pass `--custom-dataset-type mooncake_trace`, but only as a *file format* for deterministic fixed-order replay of our own prompt files (see `docs/scenarios/README.md`). This notebook uses the **actual Mooncake dataset with real timestamps** — a different thing.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). The office-hours gist pinned `release/0.3.0`; pin whatever version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.
- Internet access to download the trace (~23,600 requests, one-time).
- **Context-window caution:** the trace's median ISL is ~6.4K tokens with a long tail past 100K. Requests longer than your endpoint's context limit will fail — the gist's demo lost 4% of requests to a 32K limit. Expect some errors unless you filter the trace or serve a long-context model.

In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH — install it before running the Test run section.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — the Mooncake trace

Before throwing a trace at an endpoint, understand what you're replaying — the workload shape determines which metrics will be stressed and how to read the results. This section is a small EDA: **3.1** what the data looks like, **3.2** statistics, **3.3** prefix-reuse structure, **3.4** how much real load it represents in human terms, **3.5** cutting the demo slice.

(Headline numbers quoted in the text below were recomputed from the downloaded trace and match the gist's within rounding.)

In [ ]:
import urllib.request

TRACE_URL = (
    "https://raw.githubusercontent.com/kvcache-ai/Mooncake/refs/heads/main/"
    "FAST25-release/arxiv-trace/mooncake_trace.jsonl"
)
trace_path = REPO_ROOT / "notebooks" / "data" / "mooncake_trace.jsonl"
trace_path.parent.mkdir(parents=True, exist_ok=True)

if not trace_path.exists():
    print(f"Downloading {TRACE_URL} ...")
    urllib.request.urlretrieve(TRACE_URL, trace_path)
print(f"Trace: {trace_path} ({trace_path.stat().st_size / 1e6:.1f} MB)")


### 3.1 What the data looks like

Four columns, one row per request — and deliberately **no prompt text**:

| Column | Type | Meaning |
|---|---|---|
| `timestamp` | int, ms from trace start | when the request arrived in production |
| `input_length` | int, tokens | ISL of the original (hidden) prompt |
| `output_length` | int, tokens | OSL of the original response |
| `hash_ids` | list[int] | the prompt's content, as IDs of ~512-token blocks |

`hash_ids` is the privacy trick: the real text is replaced by block identifiers, so a follow-up question about the same paper shares the leading `hash_ids` of the earlier request. Content is gone, **reuse structure survives** — AIPerf regenerates natural language per block ID, so identical IDs expand to identical text and a caching server sees real prefix hits. Note the first records below: three different requests all start with block `0` (a shared system-prompt/preamble block).

In [ ]:
import pandas as pd

trace = pd.read_json(trace_path, lines=True)
print(f"{len(trace)} requests | columns: {list(trace.columns)}")
print(trace.dtypes.to_string(), "\n")

sample = trace.head(5).copy()
sample["n_blocks"] = sample["hash_ids"].apply(len)
sample["hash_ids"] = sample["hash_ids"].apply(
    lambda ids: str(ids[:6])[:-1] + ", ...]" if len(ids) > 6 else str(ids)
)

# First 5 raw trace records (hash_ids truncated for display) plus a derived
# n_blocks column — a look at the trace's actual shape before replaying it.
sample


### 3.2 Statistics

Headline numbers (recomputed): **23,608 requests over 60.0 minutes** (~394 req/min, 6.6 req/s sustained). The two length distributions have completely different shapes:

- **ISL is large and long-tailed**: median 6,345 tokens, mean 8,590, p99 ~62K, max ~125K. Half the requests carry 3K–7.5K tokens of context.
- **OSL is small and bimodal**: median just **30 tokens**, yet p90 is ~507. About half of all requests produce ≤32 output tokens (short extractive answers), while a second mode sits near ~500 (longer summary-style generations, likely capped). Mean 182.

This asymmetry matters for benchmarking: the workload hammers **prefill** (TTFT, queueing) far more than **decode** (ITL) — judge the endpoint accordingly.

In [ ]:
dur_min = (trace.timestamp.max() - trace.timestamp.min()) / 60_000
rate_s = len(trace) / (dur_min * 60)
print(f"Duration: {dur_min:.1f} min | {len(trace)} requests | {len(trace)/dur_min:.0f} req/min ({rate_s:.2f} req/s)")
print(f"Means: ISL {trace.input_length.mean():.0f} tokens, OSL {trace.output_length.mean():.0f} tokens")
print(f"Share of requests with OSL <= 32 tokens: {(trace.output_length <= 32).mean()*100:.0f}%")

q = [0.01, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
summary = pd.DataFrame({
    "input_length (ISL)": trace.input_length.quantile(q).round().astype(int),
    "output_length (OSL)": trace.output_length.quantile(q).round().astype(int),
})
summary.index = [f"p{int(x*100)}" for x in q]

# ISL/OSL percentiles across the whole trace (rows = percentile, columns =
# input/output length in tokens) — the numeric backing for the 3.2 narrative above.
summary


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

trace.input_length.plot.hist(bins=80, ax=axes[0][0], log=True)
axes[0][0].set_title("ISL distribution (log count) — long tail past 100K")
axes[0][0].set_xlabel("input tokens")

trace.output_length.plot.hist(bins=80, ax=axes[0][1], log=True)
axes[0][1].set_title("OSL distribution (log count) — bimodal: short answers + ~500-token mode")
axes[0][1].set_xlabel("output tokens")

(trace.timestamp // 60_000).value_counts().sort_index().plot(ax=axes[1][0])
axes[1][0].set_title("Request rate over the hour")
axes[1][0].set_xlabel("minute")
axes[1][0].set_ylabel("requests/min")

trace.groupby(trace.timestamp // 60_000).input_length.sum().div(1e6).plot(ax=axes[1][1])
axes[1][1].set_title("Offered prefill load over the hour")
axes[1][1].set_xlabel("minute")
axes[1][1].set_ylabel("input Mtokens/min")

plt.tight_layout()


### 3.3 Prefix-reuse structure

Replaying the trace in order and asking, for each request, "how many of its blocks appeared in *any* earlier request?" quantifies the KV-cache opportunity:

- **55% of all block references repeat an earlier block** (~183K unique blocks behind ~409K references).
- **~74% of requests arrive with at least half their prefix already seen**; almost none are entirely novel.

That is a huge cache-hit potential — and precisely what synthetic random prompts have 0% of. On a server with prefix caching / KV-aware routing, most of the giant ISLs in 3.2 don't need full prefill. This is why the trace is the right tool for A/B-testing KV-cache optimizations, and why TTFT measured *with* this trace can beat TTFT from a synthetic run at the same nominal ISL.

In [ ]:
seen, hit_ratios, reused, total = set(), [], 0, 0
for ids in trace["hash_ids"]:
    hits = sum(1 for h in ids if h in seen)
    reused += hits
    total += len(ids)
    hit_ratios.append(hits / len(ids) if len(ids) else 0.0)
    seen.update(ids)
trace["prefix_hit_ratio"] = hit_ratios

block_size = (trace.input_length / trace.hash_ids.apply(len)).median()
print(f"Inferred block size: ~{block_size:.0f} tokens/block (nominal 512; the last block of a request is partial)")
print(f"Unique blocks: {len(seen):,} | block references: {total:,} | repeats: {reused:,} ({100*reused/total:.1f}%)")
print(f"Requests with >=50% of prefix already seen: {(trace.prefix_hit_ratio >= 0.5).mean()*100:.0f}%")
print(f"Requests with a fully-cached prefix:        {(trace.prefix_hit_ratio == 1.0).mean()*100:.1f}%")
print(f"Requests with an entirely novel prefix:     {(trace.prefix_hit_ratio == 0.0).mean()*100:.1f}%")

ax = trace.prefix_hit_ratio.plot.hist(bins=40, figsize=(7, 3.5))
ax.set_title("Per-request prefix cache-hit potential")
ax.set_xlabel("fraction of the request's blocks already seen in an earlier request")
plt.tight_layout()


### 3.4 How much real load does this represent?

Translating tokens into user-experience terms (≈0.75 English words/token, ≈500 words/page):

- **One median request ≈ asking a question about ~10 pages of a paper** (6,345 tokens ≈ 4,800 words); the mean is ~13 pages, the worst case ~190 pages.
- **The median answer is one sentence** (~30 tokens ≈ 22 words) — extractive Q&A; the p90 answer is a ~380-word summary.
- **System-wide, the hour of traffic offers ~56K prefill tokens/s sustained** against only ~1.2K decode tokens/s — a **~47:1 prefill:decode ratio**. This is a document-Q&A service, not a chatbot: capacity planning is about prefill compute and KV memory, not generation speed.
- **Arrivals are bursty, not smooth**: many requests land in the same millisecond (median inter-arrival ≈ 0 ms, max gap ~3 s) — with `--fixed-schedule` your endpoint gets those micro-bursts, which is exactly what a smoothed synthetic arrival rate hides.
- **Implied concurrency** (Little's law, concurrency = arrival rate × latency): at 6.6 req/s, an endpoint averaging 2 s per request holds ~13 requests in flight; at 10 s, ~66. That's the knob to compare against your `--concurrency` intuition from synthetic runs.

In [ ]:
WORDS_PER_TOKEN = 0.75   # rough English average
WORDS_PER_PAGE = 500

dur_s = dur_min * 60
tot_in, tot_out = trace.input_length.sum(), trace.output_length.sum()

print("Offered load (what the serving stack must absorb):")
print(f"  prefill: {tot_in/dur_s:>8,.0f} tokens/s sustained for an hour  ({tot_in/1e6:.0f}M input tokens total)")
print(f"  decode:  {tot_out/dur_s:>8,.0f} tokens/s                        ({tot_out/1e6:.1f}M output tokens total)")
print(f"  prefill:decode ratio ~ {tot_in/tot_out:.0f}:1")

print("\nWhat one request means to a user:")
for label, v in [("median", trace.input_length.median()), ("mean", trace.input_length.mean()),
                 ("max", trace.input_length.max())]:
    w = v * WORDS_PER_TOKEN
    print(f"  {label:>6} input: {v:>7,.0f} tokens ~ {w:>7,.0f} words ~ {w/WORDS_PER_PAGE:>4,.0f} pages of context")
for label, v in [("median", trace.output_length.median()), ("p90", trace.output_length.quantile(0.9))]:
    print(f"  {label:>6} output: {v:>6,.0f} tokens ~ {v*WORDS_PER_TOKEN:>6,.0f} words")

rate_s = len(trace) / dur_s
print("\nImplied concurrency (Little's law: in-flight = arrival rate x request latency):")
for lat in (2, 5, 10):
    print(f"  if a request takes ~{lat:>2}s end-to-end -> ~{rate_s*lat:>4,.0f} requests in flight on average")

ia = trace.timestamp.diff().dropna()
print(f"\nBurstiness: median inter-arrival {ia.median():.0f} ms (same-ms bursts), "
      f"mean {ia.mean():.0f} ms, max gap {ia.max()/1000:.1f} s")


### 3.5 Cutting the demo slice

The gist's demo workload: the **first 5 minutes, timestamps compressed 5×**, so it replays in ~1 minute. Note what speeding up preserves and distorts: prefix-reuse structure and relative burstiness survive, but the *offered rate* is 5× production — fine for a demo/capacity probe, not a faithful UX measurement.

In [ ]:
# Cut the gist's demo workload: first 5 minutes, timestamps compressed 5x -> ~1 minute replay.
SLICE_MINUTES = 5
SPEEDUP = 5

t0 = trace["timestamp"].min()
sliced = trace[trace["timestamp"] - t0 < SLICE_MINUTES * 60_000].copy()
sliced["timestamp"] = ((sliced["timestamp"] - t0) / SPEEDUP).astype(int)

slice_path = trace_path.with_name(f"mooncake_trace_{SLICE_MINUTES}min_{SPEEDUP}x.jsonl")
sliced.to_json(slice_path, orient="records", lines=True)
print(f"{len(sliced)} requests -> {slice_path.name} (replays in ~{SLICE_MINUTES * 60 // SPEEDUP}s)")


## 4. Test run

Two modes, per the gist:

- **With `--fixed-schedule`** — requests fire on the trace's timestamps: end-to-end system testing under the real arrival pattern (used below).
- **Without it** — the trace replays as fast as possible: capacity testing that still keeps the naturalistic prefix-reuse structure.

In [ ]:
def run_aiperf(cmd):
    """Print and run an aiperf command from the repo root, streaming output into the notebook."""
    print("$ " + " \\\n    ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\nexit code: {result.returncode}")
    return result

# ---- Required ----------------------------------------------------------
URL = ""         # e.g. "http://localhost:8000"
MODEL = ""       # leave empty to auto-detect from {URL}/v1/models; set to override

TOKENIZER = ""   # empty = use MODEL
FIXED_SCHEDULE = True
# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc3-mooncake"

assert URL, "Set URL above."

# Ask the endpoint what it serves — verifies the URL is reachable and spares
# the user a second copy-paste. Multi-model servers: first model wins.
if not MODEL:
    models_url = URL.rstrip("/") + "/v1/models"
    with urllib.request.urlopen(models_url, timeout=10) as resp:
        served = json.load(resp)["data"]
    assert served, f"{models_url} is reachable but lists no models."
    if len(served) > 1:
        print(f"{models_url} lists {len(served)} models: {[m['id'] for m in served]}")
    MODEL = served[0]["id"]
print(f"URL   : {URL}")
print(f"MODEL : {MODEL}")

cmd = [
    "aiperf", "profile",
    "--model", MODEL,
    "--url", URL,
    "--endpoint-type", "chat",
    "--streaming",
    "--input-file", str(slice_path),
    "--custom-dataset-type", "mooncake_trace",
    "--tokenizer", TOKENIZER or MODEL,
    "--artifact-dir", OUTPUT_DIR,
]
if FIXED_SCHEDULE:
    cmd.append("--fixed-schedule")
run_aiperf(cmd)


## 5. Results analysis

Unlike the synthetic run, ISL/OSL now *vary per request*, so their distributions appear in the summary alongside the latency metrics. Gist reference numbers for this exact workload (5-min slice, 5× speed, overprovisioned endpoint): TTFT avg ≈ 407 ms, request throughput ≈ 26.7 req/s, 96% success (the 4% failures exceeded the 32K context limit).

In [ ]:
from io import StringIO

import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
csv_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
assert csv_path is not None, f"No profile_export_aiperf.csv under {artifact_dir} — did the Test run complete?"


def read_export(path):
    """Split the multi-section AIPerf CSV into (stats, totals) DataFrames.

    Section 1: per-request statistics (Metric, avg, min, max, p50, ..., std).
    Section 2: single-value run totals (Metric, Value).
    Section 3 (GPU telemetry, if present) is ignored here.
    """
    sections = Path(path).read_text().strip().split("\n\n")
    stats = pd.read_csv(StringIO(sections[0]))
    totals = (pd.read_csv(StringIO(sections[1]))
              if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"]))
    return stats, totals


stats, totals = read_export(csv_path)
pd.set_option("display.max_rows", None)

# Section 1: per-request statistics — one row per metric, columns are
# avg/min/max/percentiles/std across all requests in the run. Unlike the
# synthetic use case, ISL/OSL rows now show real per-request variation.
print("Per-request statistics (one row per metric, columns = avg/min/max/percentiles/std):")
display(stats)

# Section 2: run totals — single-value rows (duration, request count,
# system-wide throughput, error/success counts).
print("Run totals (single-value rows: duration, request count, system-wide throughput):")
totals


In [ ]:
# Latency and per-request length distributions (statistics section) — with a trace,
# ISL/OSL vary per request, so their distributions are informative here.
key = ["time to first token", "request latency", "inter token latency",
       "input sequence length", "output sequence length"]
mask = stats["Metric"].str.lower().apply(lambda n: any(p in n for p in key))
print("Latency and per-request ISL/OSL — avg/p50/p90/p99 across all requests:")
display(stats.loc[mask, ["Metric", "avg", "p50", "p90", "p99"]])

# Run totals: throughput, request count, duration, and any error counters.
print("Run totals — throughput, request count, duration, error counters:")
totals


### Notes — A/B testing with a trace

- To measure a KV-cache optimization: run this identical command against endpoint A (optimization off) and endpoint B (on), and compare TTFT. With ~real prefix reuse, cache hits skip prefill work and TTFT drops; Dynamo's repo documents a smart-router A/B using exactly this method.
- Scale the load by editing the trace (e.g. duplicate every record 4× on the same schedule) — AIPerf replays whatever you give it.
- Your own saved production traces work the same way (see AIPerf's docs for accepted formats); ShareGPT is also supported.
- Errors from over-limit requests are reported in the export — check the error-rate rows before comparing latency numbers across runs.